# 🗣️ SadTalker Studio - Tạo MC Ảo Nói Chuyện Từ 1 Ảnh Tĩnh (Đã Vá Lỗi NumPy 2.0 & Caching Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Thời gian chạy (Runtime)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm **Chạy tất cả (Run all)** -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ tải model và lưu vào Drive, **từ lần thứ 2 trở đi sẽ nạp tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title 1. Cài đặt Môi trường, Tải Model Checkpoints & Vá Lỗi Tương Thích (Chạy trong 1-2 phút)
import os
import time
import shutil
from google.colab import drive
from IPython.display import clear_output

print("🔗 Đang kết nối với Google Drive của bạn...")
drive.mount('/content/drive')

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/SadTalker"
drive_checkpoints_dir = f"{drive_cache_dir}/checkpoints"
output_drive_dir = "/content/drive/MyDrive/AI_Colab_Cache/SadTalker_Outputs"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)
os.makedirs(output_drive_dir, exist_ok=True)

%cd /content
if not os.path.exists("/content/SadTalker"):
    !git clone -b v1.1 https://github.com/camenduru/SadTalker /content/SadTalker

%cd /content/SadTalker
!pip install -q gradio safetensors kornia facexlib yacs gfpgan

# Vá lỗi tương thích NumPy 2.0
!sed -i 's/category=np.VisibleDeprecationWarning/category=Warning/g' /content/SadTalker/src/face3d/util/preprocess.py

# Vá lỗi PyTorch 2.6 toàn cục qua sitecustomize.py
import site
for p in site.getsitepackages():
    sc_file = os.path.join(p, 'sitecustomize.py')
    with open(sc_file, 'w', encoding='utf-8') as f:
        f.write('''import torch
_old_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _old_torch_load(*args, **kwargs)
torch.load = _safe_torch_load
''')
    break

# Kiểm tra xem checkpoints đã có trên Drive chưa
if os.path.exists(drive_checkpoints_dir) and os.path.isdir(drive_checkpoints_dir) and len(os.listdir(drive_checkpoints_dir)) > 2:
    print("🎉 ĐÃ TÌM THẤY MODEL CHECKPOINTS TRONG GOOGLE DRIVE! Nạp trực tiếp không cần tải lại...")
    if not os.path.exists("/content/SadTalker/checkpoints"):
        !cp -r "{drive_checkpoints_dir}" /content/SadTalker/checkpoints
else:
    print("⏳ Đang tải trọng số Model SadTalker siêu tốc qua aria2c...")
    !apt -y install -qq aria2
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_256.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_256.safetensors
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/SadTalker_V0.0.2_512.safetensors -d /content/SadTalker/checkpoints -o SadTalker_V0.0.2_512.safetensors
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00109-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00109-model.pth.tar
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/checkpoints/mapping_00229-model.pth.tar -d /content/SadTalker/checkpoints -o mapping_00229-model.pth.tar
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/GFPGANv1.4.pth -d /content/SadTalker/gfpgan/weights -o GFPGANv1.4.pth
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/alignment_WFLW_4HG.pth -d /content/SadTalker/gfpgan/weights -o alignment_WFLW_4HG.pth
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/detection_Resnet50_Final.pth -d /content/SadTalker/gfpgan/weights -o detection_Resnet50_Final.pth
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/SadTalker/resolve/main/new/gfpgan/weights/parsing_parsenet.pth -d /content/SadTalker/gfpgan/weights -o parsing_parsenet.pth
    print("💾 Đang lưu bản sao model vào Google Drive để lần sau không phải tải lại...")
    os.makedirs(drive_cache_dir, exist_ok=True)
    !cp -r /content/SadTalker/checkpoints "{drive_cache_dir}/"

# Khởi tạo Cloudflare Tunnel dự phòng
print("🌐 Đang khởi tạo Cloudflare Tunnel công khai...")
!curl -LOs https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
!nohup cloudflared tunnel --url http://127.0.0.1:7860 > /content/cloudflared_sadtalker.log 2>&1 &

time.sleep(4)
cf_link = ""
if os.path.exists("/content/cloudflared_sadtalker.log"):
    import re
    with open("/content/cloudflared_sadtalker.log", "r", encoding="utf-8", errors="ignore") as f:
        matches = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
        if matches:
            cf_link = matches[0]

clear_output()
print("=" * 65)
print("🎉 Cài đặt & Vá lỗi hoàn tất! Đang khởi động WebUI SadTalker...")
if cf_link:
    print(f"🔗 LINK CLOUDFLARE (TRUY CẬP NGAY): {cf_link}")
print("🔗 LINK GRADIO LIVE SẼ XUẤT HIỆN NGAY BÊN DƯỚI:")
print("=" * 65)

!python -u app_sadtalker.py
